# Modelo Preditivo de Classificação de NPS

Para um modelo de classificação precisamos transformar a variável target numérica `nps_score` em uma variável categórica.
Podemos utilizar tanto a classificação multiclasse já conhecida do NPS como `Promoter`, `Passive`, `Detractor`, ou também podemos simplificar para uma classificação binária `Satisfied` = [`Promoter`, `Passive`] e `Unsatisfied` = [`Detractor`].
Iremos treinar e testar ambas classificações.

In [48]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


warnings.filterwarnings('ignore')

In [27]:
df = pd.read_csv('../data/desafio_nps_fase_1.csv')
df.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


## Transformação de Variáveis Target

In [28]:
# Classificação NPS em 2 classes:
def classify_nps(score):
    if score >= 7:
        return 'Satisfied'
    return 'Unsatisfied'

df['nps_class'] = df['nps_score'].apply(classify_nps)
df[['nps_score', 'nps_class']]

,nps_score,nps_class
0,6.9,Unsatisfied
1,2.4,Unsatisfied
2,4.8,Unsatisfied
3,5.9,Unsatisfied
4,6.1,Unsatisfied
...,...,...
2495,3.7,Unsatisfied
2496,3.7,Unsatisfied
2497,7.4,Satisfied
2498,2.3,Unsatisfied


## Seleção e Sepração de Dados

In [29]:
# Configuração de validação cruzada compartilhada para todos os modelos
shared_cv = KFold(n_splits=10, shuffle=True, random_state=42)

# One Hot Encode a coluna 'customer_region'
one_hot_encoded_regions = pd.get_dummies(df['customer_region'], prefix='region', drop_first=True)

# Features de treinamento
X = df.drop(columns=[
    'customer_id',
    'order_id',
    'customer_region',
    'nps_score',
    'nps_class'])

# Join das features com as colunas one-hot encoded
X = X.join(one_hot_encoded_regions)
y = df['nps_class'] # Labels resultado binário

# Divisão dos dados em treino e teste para ambos os conjuntos de labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Logistic Regression

In [43]:
logistic_regression = LogisticRegression(random_state=42)
logistic_regression.fit(X_train, y_train)
logistic_regression_predictions = logistic_regression.predict(X_test)
print(classification_report(logistic_regression_predictions, y_test))

              precision    recall  f1-score   support

   Satisfied       0.38      0.69      0.49        67
 Unsatisfied       0.97      0.89      0.93       683

    accuracy                           0.87       750
   macro avg       0.67      0.79      0.71       750
weighted avg       0.91      0.87      0.89       750



## Decision Tree

In [44]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)
decision_tree_predictions = decision_tree.predict(X_test)
print(classification_report(decision_tree_predictions, y_test))

              precision    recall  f1-score   support

   Satisfied       0.68      0.59      0.63       140
 Unsatisfied       0.91      0.94      0.92       610

    accuracy                           0.87       750
   macro avg       0.79      0.76      0.78       750
weighted avg       0.87      0.87      0.87       750



# Random Forest

In [45]:
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train, y_train)
random_forest_predictions = random_forest.predict(X_test)
print(classification_report(random_forest_predictions, y_test))

              precision    recall  f1-score   support

   Satisfied       0.60      0.94      0.73        78
 Unsatisfied       0.99      0.93      0.96       672

    accuracy                           0.93       750
   macro avg       0.80      0.93      0.84       750
weighted avg       0.95      0.93      0.93       750



In [ ]:
logistic_regression_scores = cross_val_score(logistic_regression, X_train, y_train, cv=shared_cv)
decision_tree_scores = cross_val_score(decision_tree, X_train, y_train, cv=shared_cv)
random_forest_scores = cross_val_score(random_forest, X_train, y_train, cv=shared_cv)

cv_scores = pd.DataFrame(
    index=['Logistic Regression', 'Decision Tree', 'Random Forest'],
    data = {
        'scores': [logistic_regression_scores, decision_tree_scores, random_forest_scores],
        'mean': [np.mean(logistic_regression_scores), np.mean(decision_tree_scores), np.mean(random_forest_scores)],
        'std': [np.std(logistic_regression_scores), np.std(decision_tree_scores), np.std(random_forest_scores)]
    }
)

cv_scores

,scores,mean,std
Logistic Regression,"[0.8857142857142857, 0.8514285714285714, 0.914...",0.888571,0.027075
Decision Tree,"[0.9085714285714286, 0.8742857142857143, 0.908...",0.889143,0.022886
Random Forest,"[0.92, 0.9142857142857143, 0.9371428571428572,...",0.929714,0.015129


## Metrificação